In [2]:
import importlib
import sys

# Remove cached modules so fresh versions load
mods_to_remove = [k for k in sys.modules if 'retrieval' in k or 'evaluation' in k or 'hallucination' in k]
for mod in mods_to_remove:
    del sys.modules[mod]

# Now import fresh
from retrieval.rag_pipeline import RAGPipeline

rag = RAGPipeline()
result = rag.query("What technologies did the candidate use?")

print(result["confidence_report"])

print("\n📄 CITED ANSWER:")
print(result["cited_answer"])

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ VectorStore ready — collection: truthguard_docs
   Documents in store: 16
✅ CitationMapper ready — using BAAI/bge-small-en-v1.5
✅ HallucinationDetector ready
✅ RAG Pipeline ready — using phi3:mini
  TRUTHGUARD CONFIDENCE REPORT
  Trust Level     : ⚠️ MEDIUM
  Confidence Score: 63.9%
  Grounding Score : 66.7%
  Avg Retrieval   : 57.6%
--------------------------------------------------
  Sentences
    ✅ Supported  : 1
    ⚠️  Weak       : 2
    ❌ Unsupported : 0
--------------------------------------------------
  💡 Answer is mostly supported. Verify weak sentences.

📄 CITED ANSWER:
The candidate utilized a variety of technologies including Databricks MLflow for machine learning model management, Databricks SQL / Delta Tables along with PySpark for data processing and analytics. [1] For full-stack application development, the candidate employed React.js as well as Node.js frameworks such as Express or FastAPI coupled with Uvicorn server runtime on top of Python ML libraries like scikit

In [3]:
from evaluation.ragas_evaluator import RAGASEvaluator

evaluator = RAGASEvaluator()

# Build a small eval dataset from your resume PDF
eval_pairs = [
    {
        "question": "What technologies did the candidate use?",
        "answer": result["answer"],
        "contexts": [c["text"] for c in result["sources"]]
    },
    {
        "question": "What internships has the candidate done?",
        "answer": rag.query("What internships has the candidate done?")["answer"],
        "contexts": [c["text"] for c in result["sources"]]
    },
    {
        "question": "What machine learning projects has the candidate worked on?",
        "answer": rag.query("What machine learning projects has the candidate worked on?")["answer"],
        "contexts": [c["text"] for c in result["sources"]]
    }
]

print("Running RAGAS evaluation — this takes a few minutes...")
ragas_results = evaluator.evaluate_dataset(eval_pairs)

agg = ragas_results["aggregate"]
print(f"\n📊 RAGAS AGGREGATE SCORES")
print(f"  Faithfulness      : {agg['avg_faithfulness']}")
print(f"  Answer Relevancy  : {agg['avg_answer_relevancy']}")
print(f"  Context Precision : {agg['avg_context_precision']}")
print(f"  Evaluated         : {agg['total_evaluated']} questions")

✅ RAGAS Evaluator ready — using local LLM + embeddings
Running RAGAS evaluation — this takes a few minutes...


Evaluating:   0%|                                                                                | 0/3 [00:00<?, ?it/s]Exception raised in Job[0]: TimeoutError()
Exception raised in Job[2]: TimeoutError()
Evaluating: 100%|████████████████████████████████████████████████████████████████████████| 3/3 [03:00<00:00, 60.01s/it]


  Evaluated: What technologies did the candidate use?...


Evaluating:  33%|███████████████████████▋                                               | 1/3 [02:23<04:46, 143.27s/it]Exception raised in Job[2]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
Evaluating: 100%|████████████████████████████████████████████████████████████████████████| 3/3 [03:00<00:00, 60.01s/it]


  Evaluated: What internships has the candidate done?...


Evaluating:  33%|███████████████████████▋                                               | 1/3 [03:00<06:00, 180.01s/it]Exception raised in Job[2]: TimeoutError()
Exception raised in Job[1]: TimeoutError()
Evaluating: 100%|████████████████████████████████████████████████████████████████████████| 3/3 [03:00<00:00, 60.03s/it]


  Evaluated: What machine learning projects has the candidate w...

📊 RAGAS AGGREGATE SCORES
  Faithfulness      : nan
  Answer Relevancy  : nan
  Context Precision : nan
  Evaluated         : 3 questions


In [6]:
from evaluation.ragas_evaluator import RAGASEvaluator

evaluator = RAGASEvaluator()

# Test with just one pair first to keep it fast
single_result = evaluator.evaluate_single(
    question="What technologies did the candidate use?",
    answer=result["answer"],
    contexts=[c["text"] for c in result["sources"]]
)

print(f"Faithfulness      : {single_result['faithfulness']}")
print(f"Answer Relevancy  : {single_result['answer_relevancy']}")
print(f"Context Precision : {single_result['context_precision']}")

✅ RAGAS Evaluator ready — using local LLM + embeddings


Evaluating:   0%|                                                                                | 0/3 [00:00<?, ?it/s]Exception raised in Job[0]: TimeoutError()
Exception raised in Job[2]: TimeoutError()
Evaluating: 100%|████████████████████████████████████████████████████████████████████████| 3/3 [03:00<00:00, 60.00s/it]


Faithfulness      : nan
Answer Relevancy  : nan
Context Precision : nan


In [7]:
# After updating rags_evaluator

import sys
mods_to_remove = [k for k in sys.modules if 'evaluation' in k]
for mod in mods_to_remove:
    del sys.modules[mod]

from evaluation.ragas_evaluator import RAGASEvaluator

evaluator = RAGASEvaluator()

single_result = evaluator.evaluate_single(
    question="What technologies did the candidate use?",
    answer=result["answer"],
    contexts=[c["text"] for c in result["sources"]]
)

print(f"Faithfulness      : {single_result['faithfulness']}")
print(f"Answer Relevancy  : {single_result['answer_relevancy']}")
print(f"Context Precision : {single_result['context_precision']}")

✅ RAGAS Evaluator ready — using local LLM + embeddings


Evaluating: 100%|███████████████████████████████████████████████████████████████████████| 1/1 [03:00<00:00, 180.01s/it]


Faithfulness      : nan
Answer Relevancy  : 0.7883
Context Precision : nan


In [2]:
import sys
mods_to_remove = [k for k in sys.modules if any(x in k for x in ['retrieval', 'evaluation', 'hallucination', 'ingestion'])]
for mod in mods_to_remove:
    del sys.modules[mod]

from retrieval.rag_pipeline import RAGPipeline

rag = RAGPipeline()
result = rag.query("What technologies did the candidate use?")
print("✅ result ready")
print(f"Answer preview: {result['answer'][:100]}...")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ VectorStore ready — collection: truthguard_docs
   Documents in store: 16
✅ CitationMapper ready — using BAAI/bge-small-en-v1.5


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


✅ HallucinationDetector ready
✅ RAG Pipeline ready — using phi3:mini
✅ result ready
Answer preview: The candidate utilized a variety of technologies including Python, Java, React.js, Node.js, Databric...


In [4]:
#  After changing the model from phi3:mini to gemma3:1b

import sys
mods_to_remove = [k for k in sys.modules if 'evaluation' in k]
for mod in mods_to_remove:
    del sys.modules[mod]

from evaluation.ragas_evaluator import RAGASEvaluator

evaluator = RAGASEvaluator()

single_result = evaluator.evaluate_single(
    question="What technologies did the candidate use?",
    answer=result["answer"],
    contexts=[c["text"] for c in result["sources"]]
)

print(f"Faithfulness      : {single_result['faithfulness']}")
print(f"Answer Relevancy  : {single_result['answer_relevancy']}")
print(f"Context Precision : {single_result['context_precision']}")

✅ RAGAS Evaluator ready — using gemma3:1b for evaluation


Evaluating: 100%|████████████████████████████████████████████████████████████████████████| 1/1 [00:23<00:00, 23.56s/it]


Faithfulness      : 0.9231
Answer Relevancy  : 0.0
Context Precision : 1.0
